##### date_trunc()

- is used to truncate a **timestamp** to a specified unit of time (e.g., **year, month, day, hour**).
- To `truncate` at **Year, Month, Day, Hour, Minute and Seconds** units and returns `Date` in Spark DateType format **yyyy-MM-dd HH:mm:ss.SSSS**.

##### Difference Between trunc() and date_trunc()

| Function       | Works On  | Returns                                   |
| -------------- | --------- | ----------------------------------------- |
| `trunc()`      | Date      | First day of month/year/quarter/week                 |
| `date_trunc()` | Timestamp | Truncated timestamp (hour/day/month etc.) |

**Syntax**

     date_trunc(format, timestamp)

- **format:** literal string

|       format       |      Description                                |      Example                  |
|--------------------|-------------------------------------------------|-------------------------------|      
| `year, yyyy, yy`   | Truncate to the **first day of the year**.      |  `date_trunc("year", col)`    |
| `month, mon, mm`   | Truncate to the **first day of the month**.     |  `date_trunc("month", col)`   |
| `day, dd`          | **Zero out the time part (set to midnight)**.   |  `date_trunc("day", col)`     |
| `quarter`          | Truncate to the **first day of the quarter**.   |  `date_trunc('quarter', col)` |
| `week`             | Truncate to the **Monday of the week**.         |  `date_trunc('week', col)`    |
| `hour`             | **Zero out minutes and seconds**.               |  `date_trunc("hour", col)`    |
| `minute`           | **Zero out seconds**.                           |  `date_trunc("minute", col)`  |
| `second`           | **Zero out fractional seconds**.                |  `date_trunc('second', col)`  |
| `microsecond`      |   |  `date_trunc('microsecond', col)` |
| `millisecond`      |   |  `date_trunc('millisecond', col)` |

- **Input Type:**
  - The function expects a **TimestampType column** as **input**.
  - If your data is a **string**, you may need to **cast** it to a **timestamp first** using **col("column_name").cast("timestamp")**.

- **Return Type:**
  - It returns a **TimestampType.**
  - If you need the result in a **specific string format** (e.g., **"YYYY-MM" without the day and time**), you should wrap the result in the **date_format()** function.

     second      → 10:15:30
     millisecond → 10:15:30.123
     microsecond → 10:15:30.123456

| Function                         | Precision kept          | Result               |
| -------------------------------- | ----------------------- | -------------------- |
| `date_trunc('millisecond', col)` | milliseconds (3 digits) | removes microseconds |
| `date_trunc('microsecond', col)` | microseconds (6 digits) | usually unchanged    |

**Input: 2025-03-11 10:15:30.987654**

| Function                    | Output                       |
| --------------------------- | ---------------------------- |
| `date_trunc('millisecond')` | `2025-03-11 10:15:30.987`    |
| `date_trunc('microsecond')` | `2025-03-11 10:15:30.987654` |

In [0]:
from pyspark.sql.functions import date_trunc, col, current_timestamp, to_timestamp, date_format

**a) Year, Month, Day, Hour, Minute, Second, Week, quarter**
- trun() -> Year, Month, Week, quarter
- date_trunc() -> Year, Month, Day, Hour, Minute, Second, Week, quarter

In [0]:
spark.range(1).select(
    current_timestamp().alias("today"),
    date_trunc("Year", current_timestamp()).alias("Year"),
    date_trunc("Month", current_timestamp()).alias("Month"),
    date_trunc("Day", current_timestamp()).alias("Day"),
    date_trunc("Hour", current_timestamp()).alias("Hour"),
    date_trunc("Minute", current_timestamp()).alias("Minute"),
    date_trunc("Second", current_timestamp()).alias("Second"),
    date_trunc("Week", current_timestamp()).alias("Week"),
    date_trunc("Quarter", current_timestamp()).alias("Quarter"),
    date_trunc("microsecond", current_timestamp()).alias("microsecond"),
    date_trunc("millisecond", current_timestamp()).alias("millisecond")
).display()


today,Year,Month,Day,Hour,Minute,Second,Week,Quarter,microsecond,millisecond
2026-03-11T17:56:17.705Z,2026-01-01T00:00:00.000Z,2026-03-01T00:00:00.000Z,2026-03-11T00:00:00.000Z,2026-03-11T17:00:00.000Z,2026-03-11T17:56:00.000Z,2026-03-11T17:56:17.000Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T17:56:17.705Z,2026-03-11T17:56:17.705Z


In [0]:
df_ts = spark.createDataFrame([("2026-02-22 14:35:50.123",),
                               ("2026-01-11 10:15:30.123456",),
                               ("2025-09-29 11:45:55.456",),
                               ("2025-06-05 06:25:35.456789",),
                               ("2025-04-04 07:55:05.987654",)],
                              ["start_ts"]) \
    .withColumn("start_ts", to_timestamp(col("start_ts"), 'yyyy-MM-dd HH:mm:ss.SSSSSS')) \

display(df_ts)

start_ts
2026-02-22T14:35:50.123Z
2026-01-11T10:15:30.123Z
2025-09-29T11:45:55.456Z
2025-06-05T06:25:35.456Z
2025-04-04T07:55:05.987Z


Spark timestamp display **format** is fixed to **millisecond** precision in **most UIs** (including Databricks display).

|       Issue             |          format               |
|-------------------------|-------------------------------|
| Spark internally stores | 2026-02-22 14:35:50.123000    |
| But display shows       | 2026-02-22T14:35:50.123+00:00 |

In [0]:
df_ts.select("start_ts",
    date_trunc("Year", col("start_ts")).alias("Year"),
    date_trunc("Month", col("start_ts")).alias("Month"),
    date_trunc("Day", col("start_ts")).alias("Day"),
    date_trunc("Hour", col("start_ts")).alias("Hour"),
    date_trunc("Minute", col("start_ts")).alias("Minute"),
    date_trunc("Second", col("start_ts")).alias("Second"),
    date_trunc("week", col("start_ts")).alias("Week"),
    date_trunc("quarter", col("start_ts")).alias("Quarter"),
    date_trunc("microsecond", col("start_ts")).alias("microsecond"),
    date_trunc("millisecond", col("start_ts")).alias("millisecond")).display()

start_ts,Year,Month,Day,Hour,Minute,Second,Week,Quarter,microsecond,millisecond
2026-02-22T14:35:50.123Z,2026-01-01T00:00:00.000Z,2026-02-01T00:00:00.000Z,2026-02-22T00:00:00.000Z,2026-02-22T14:00:00.000Z,2026-02-22T14:35:00.000Z,2026-02-22T14:35:50.000Z,2026-02-16T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-02-22T14:35:50.123Z,2026-02-22T14:35:50.123Z
2026-01-11T10:15:30.123Z,2026-01-01T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-01-11T00:00:00.000Z,2026-01-11T10:00:00.000Z,2026-01-11T10:15:00.000Z,2026-01-11T10:15:30.000Z,2026-01-05T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-01-11T10:15:30.123Z,2026-01-11T10:15:30.123Z
2025-09-29T11:45:55.456Z,2025-01-01T00:00:00.000Z,2025-09-01T00:00:00.000Z,2025-09-29T00:00:00.000Z,2025-09-29T11:00:00.000Z,2025-09-29T11:45:00.000Z,2025-09-29T11:45:55.000Z,2025-09-29T00:00:00.000Z,2025-07-01T00:00:00.000Z,2025-09-29T11:45:55.456Z,2025-09-29T11:45:55.456Z
2025-06-05T06:25:35.456Z,2025-01-01T00:00:00.000Z,2025-06-01T00:00:00.000Z,2025-06-05T00:00:00.000Z,2025-06-05T06:00:00.000Z,2025-06-05T06:25:00.000Z,2025-06-05T06:25:35.000Z,2025-06-02T00:00:00.000Z,2025-04-01T00:00:00.000Z,2025-06-05T06:25:35.456Z,2025-06-05T06:25:35.456Z
2025-04-04T07:55:05.987Z,2025-01-01T00:00:00.000Z,2025-04-01T00:00:00.000Z,2025-04-04T00:00:00.000Z,2025-04-04T07:00:00.000Z,2025-04-04T07:55:00.000Z,2025-04-04T07:55:05.000Z,2025-03-31T00:00:00.000Z,2025-04-01T00:00:00.000Z,2025-04-04T07:55:05.987Z,2025-04-04T07:55:05.987Z


**b) Week, Quarter & millisecond**

In [0]:
# df_dtTrnc = spark.range(5)
data = [(1,), (2,), (3,), (4,), (5,)]
df_dtTrnc = spark.createDataFrame(data, ["id"])
display(df_dtTrnc)

id
1
2
3
4
5


In [0]:
df_dtTrnc_mill = df_dtTrnc.withColumn("created_ts", current_timestamp())
display(df_dtTrnc_mill)

id,created_ts
1,2026-03-11T18:19:35.903Z
2,2026-03-11T18:19:35.903Z
3,2026-03-11T18:19:35.903Z
4,2026-03-11T18:19:35.903Z
5,2026-03-11T18:19:35.903Z


In [0]:
df_dtTrnc_mill.write.mode("overwrite").saveAsTable("sample_time_table")

In [0]:
spark.sql("""
SELECT
    created_ts,
    date_trunc("Week", created_ts) AS Week,
    date_trunc('Quarter', created_ts) AS Quarter,
    date_trunc('millisecond', created_ts) AS millisecond
FROM sample_time_table
""").display()

created_ts,Week,Quarter,millisecond
2026-03-11T18:19:50.441Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T18:19:50.441Z
2026-03-11T18:19:50.441Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T18:19:50.441Z
2026-03-11T18:19:50.441Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T18:19:50.441Z
2026-03-11T18:19:50.441Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T18:19:50.441Z
2026-03-11T18:19:50.441Z,2026-03-09T00:00:00.000Z,2026-01-01T00:00:00.000Z,2026-03-11T18:19:50.441Z
